# Notebook 4: Distributed Coordination — ADMM for BRP Portfolio

## References

1. **Boyd, S., Parikh, N., Chu, E., Peleato, B., Eckstein, J. (2011).** *"Distributed Optimization and Statistical Learning via the Alternating Direction Method of Multipliers."* Foundations and Trends in Machine Learning, 3(1), 1–122. [DOI: 10.1561/2200000016](https://doi.org/10.1561/2200000016)

2. **Dall'Anese, E., Zhu, H., Giannakis, G. B. (2013).** *"Distributed optimal power flow for smart microgrids."* IEEE Transactions on Smart Grid, 4(3), 1464–1475. [DOI: 10.1109/TSG.2013.2248175](https://doi.org/10.1109/TSG.2013.2248175)

3. **Kraning, M., Chu, E., Lavaei, J., Boyd, S. (2014).** *"Dynamic network energy management via proximal message passing."* Foundations and Trends in Optimization, 1(2), 73–126. [DOI: 10.1561/2400000002](https://doi.org/10.1561/2400000002)

## What is implemented below

We demonstrate **Jacobi ADMM** for coordinating prosumer BESS scheduling within a BRP portfolio. The problem:

$$\min_{g_1,\ldots,g_N, z} \; \underbrace{c^\top z + \frac{\beta}{2}\|z\|^2}_{\text{portfolio cost}} + \sum_{i=1}^N \underbrace{\frac{\alpha_i}{2}\|g_i - d_i\|^2}_{\text{site discomfort}} \quad \text{s.t.} \quad \sum_i g_i = z$$

where $g_i$ is site $i$'s net grid exchange, $z$ is the portfolio aggregate, $d_i$ is preferred schedule, and $\alpha_i$ is the discomfort weight.

The coupling constraint $\sum_i g_i = z$ is handled by ADMM, decomposing into parallel site-level QPs and a coordinator QP.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

np.random.seed(42)


## 1. Problem Data


In [ ]:
T = 24
hours = np.arange(T)
N = 5

# Site data
d = []      # preferred net exchange (net load without battery)
flex = []   # flexibility range (battery enables deviation from d)
alpha = []  # discomfort weight (higher = less flexible)

for i in range(N):
    pv_peak = 3.0 + 2.0 * np.random.rand()
    load_base = 1.0 + 1.5 * np.random.rand()
    d_i = load_base + 0.8*np.exp(-0.5*((hours-8)/2)**2) + 1.0*np.exp(-0.5*((hours-19)/2.5)**2) \
          - pv_peak * np.exp(-0.5*((hours-12)/3)**2)
    d.append(d_i)
    flex.append(2.0 + 2.0 * np.random.rand())
    alpha.append(0.5 + 1.5 * np.random.rand())

# DA price and portfolio risk penalty
c_da = np.maximum(0.04 + 0.02*np.sin(2*np.pi*(hours-6)/24) + 0.01*np.exp(-0.5*((hours-18)/3)**2), 0.01)
beta = 0.005

print(f"Sites: {N}, T: {T}")
for i in range(N):
    print(f"  Site {i}: α={alpha[i]:.2f}, flex={flex[i]:.1f} kW")


## 2. Centralized Solution (Reference)


In [ ]:
def total_cost(g_flat):
    G = g_flat.reshape(N, T)
    z = np.sum(G, axis=0)
    port = np.dot(c_da, z) + (beta/2)*np.dot(z, z)
    local = sum(0.5*alpha[i]*np.sum((G[i]-d[i])**2) for i in range(N))
    return port + local

def total_grad(g_flat):
    G = g_flat.reshape(N, T)
    z = np.sum(G, axis=0)
    grad = np.zeros_like(G)
    for i in range(N):
        grad[i] = (c_da + beta*z) + alpha[i]*(G[i]-d[i])
    return grad.ravel()

bounds = []
for i in range(N):
    for t in range(T):
        bounds.append((d[i][t]-flex[i], d[i][t]+flex[i]))

g0 = np.concatenate(d)
res = minimize(total_cost, g0, jac=total_grad, bounds=bounds, method='L-BFGS-B')
G_opt = res.x.reshape(N, T)
z_opt = np.sum(G_opt, axis=0)
cost_opt = res.fun
print(f"Centralized cost: {cost_opt:.4f}")


## 3. Jacobi ADMM

### Augmented Lagrangian
$$L_\rho = h(z) + \sum_i f_i(g_i) + y^\top\left(\sum_i g_i - z\right) + \frac{\rho}{2}\left\|\sum_i g_i - z\right\|^2$$

### Updates

**Site $i$ update** (parallel):
$$g_i^{k+1} = \arg\min_{g_i} f_i(g_i) + \frac{\rho}{2}\left\|g_i - \tilde{g}_i^k\right\|^2$$
where $\tilde{g}_i^k = z^k - \sum_{j \neq i} g_j^k - y^k/\rho$ is the "target" for site $i$.

This has a **closed-form solution**: $g_i^{k+1} = \text{clip}\left(\frac{\alpha_i d_i + \rho \tilde{g}_i^k}{\alpha_i + \rho}, \; d_i - f_i, \; d_i + f_i\right)$

**Coordinator (z) update**:
$$z^{k+1} = \frac{\rho\left(\sum_i g_i^{k+1} + y^k/\rho\right) - c}{\beta + \rho}$$

**Dual update**: $y^{k+1} = y^k + \rho\left(\sum_i g_i^{k+1} - z^{k+1}\right)$


In [ ]:
def site_update_qp(d_i, alpha_i, flex_i, rho, target):
    """Closed-form proximal update for site QP with box constraints."""
    g = (alpha_i * d_i + rho * target) / (alpha_i + rho)
    return np.clip(g, d_i - flex_i, d_i + flex_i)

def coordinator_update_qp(c_da, beta, rho, v):
    """Closed-form proximal update for coordinator QP."""
    return (rho * v - c_da) / (beta + rho)

# ADMM parameters
rho = 1.0
max_iter = 200
tol = 1e-5

# Initialize
G = np.array(d, dtype=float)  # start at preferred schedules
S = np.sum(G, axis=0)          # running aggregate
z = S.copy()                   # portfolio variable
y = np.zeros(T)                # dual variable (single vector since constraint is aggregate)

pr_hist, dr_hist, cost_hist = [], [], []

print(f"{'k':>3} {'primal':>10} {'dual':>10} {'cost':>10} {'gap%':>8}")
print("-"*46)

for k in range(max_iter):
    z_old = z.copy()
    G_old = G.copy()
    
    # Site updates (parallel — each uses global z, aggregate S, and dual y)
    S_new = np.zeros(T)
    for i in range(N):
        # Target for site i: what it should aim for given all other sites
        target_i = z - (S - G_old[i]) - y / rho
        G[i] = site_update_qp(d[i], alpha[i], flex[i], rho, target_i)
        S_new += G[i]
    S = S_new
    
    # Coordinator update
    v = S + y / rho
    z = coordinator_update_qp(c_da, beta, rho, v)
    
    # Dual update
    y = y + rho * (S - z)
    
    # Residuals
    primal_res = np.linalg.norm(S - z)
    dual_res = rho * np.linalg.norm(z - z_old)
    cost = total_cost(G.ravel())
    
    pr_hist.append(primal_res)
    dr_hist.append(dual_res)
    cost_hist.append(cost)
    
    if k % 20 == 0 or k == max_iter-1 or (primal_res < tol and dual_res < tol):
        gap = 100 * abs(cost - cost_opt) / abs(cost_opt)
        print(f"{k:3d} {primal_res:10.6f} {dual_res:10.6f} {cost:10.4f} {gap:8.3f}")
    
    if primal_res < tol and dual_res < tol:
        print(f"\n✓ Converged at iteration {k}!")
        break

gap_final = 100 * abs(cost_hist[-1] - cost_opt) / abs(cost_opt)
print(f"\nFinal cost:      {cost_hist[-1]:.6f}")
print(f"Central cost:    {cost_opt:.6f}")
print(f"Optimality gap:  {gap_final:.4f}%")


## 4. Convergence Analysis


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0,0].semilogy(pr_hist, 'b-', lw=1.5, label='Primal residual')
axes[0,0].semilogy(dr_hist, 'r-', lw=1.5, label='Dual residual')
axes[0,0].axhline(tol, color='gray', ls='--', label=f'ε = {tol}')
axes[0,0].set_xlabel('Iteration'); axes[0,0].set_ylabel('Residual')
axes[0,0].set_title('ADMM Convergence'); axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(cost_hist, 'g-', lw=2)
axes[0,1].axhline(cost_opt, color='r', ls='--', lw=1.5, label=f'Central opt = {cost_opt:.3f}')
axes[0,1].set_xlabel('Iteration'); axes[0,1].set_ylabel('Cost')
axes[0,1].set_title('Cost → Centralized Optimum'); axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

for i in range(N):
    axes[1,0].plot(hours, G[i], '-', lw=1.5, label=f'ADMM site {i}')
    axes[1,0].plot(hours, G_opt[i], '--', lw=1, alpha=0.4)
axes[1,0].plot(hours, S, 'k-', lw=2.5, label='ADMM aggregate')
axes[1,0].plot(hours, z_opt, 'k--', lw=2, label='Central aggregate')
axes[1,0].set_xlabel('Hour'); axes[1,0].set_ylabel('kW')
axes[1,0].set_title('ADMM (solid) vs Centralized (dashed)')
axes[1,0].legend(fontsize=6); axes[1,0].grid(alpha=0.3)

# Dual variable = internal transfer price
axes[1,1].plot(hours, y*1000, 'b-o', ms=3, lw=2, label='Dual y')
axes[1,1].plot(hours, c_da*1000, 'r--', lw=1.5, label='DA price')
axes[1,1].set_xlabel('Hour'); axes[1,1].set_ylabel('EUR/MWh')
axes[1,1].set_title('ADMM Dual Variable vs DA Price')
axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

plt.tight_layout(); plt.savefig('/tmp/nb4_conv.png', dpi=100); plt.show()


## 5. Privacy and Communication


In [ ]:
n_it = len(cost_hist)
print(f"Converged in {n_it} iterations, gap = {gap_final:.4f}%")
print()
print("Communication per iteration:")
print(f"  Coordinator → all sites: z(t), S(t), y(t)/ρ  = {3*T} floats")
print(f"  Each site → coordinator: g_i(t)               = {T} floats")
print(f"  Total per iteration: {(3+N)*T*8} bytes = {(3+N)*T*8/1024:.1f} KB")
print(f"  Grand total: {(3+N)*T*8*n_it/1024:.1f} KB")
print()
print("Privacy: sites keep private PV, load, battery data, SoC, α, f")
print("Sites share ONLY: net grid exchange schedule g_i(t)")


## 6. ρ Sensitivity and Convergence Speed


In [ ]:
rho_values = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
iters_needed = []

for rho_t in rho_values:
    G_t = np.array(d, dtype=float); S_t = np.sum(G_t, axis=0)
    z_t = S_t.copy(); y_t = np.zeros(T)
    
    for k in range(1000):
        z_old_t = z_t.copy(); G_old_t = G_t.copy()
        S_t = np.zeros(T)
        for i in range(N):
            target = z_t - (np.sum(G_old_t, axis=0) - G_old_t[i]) - y_t/rho_t
            G_t[i] = site_update_qp(d[i], alpha[i], flex[i], rho_t, target)
            S_t += G_t[i]
        z_t = coordinator_update_qp(c_da, beta, rho_t, S_t + y_t/rho_t)
        y_t += rho_t * (S_t - z_t)
        if np.linalg.norm(S_t-z_t) < tol and rho_t*np.linalg.norm(z_t-z_old_t) < tol:
            iters_needed.append(k+1); break
    else:
        iters_needed.append(1000)

plt.figure(figsize=(8, 4))
plt.bar([f'{r}' for r in rho_values], iters_needed, color='steelblue', alpha=0.7)
plt.xlabel('ρ'); plt.ylabel('Iterations'); plt.title('Convergence Speed vs ρ')
plt.grid(alpha=0.3, axis='y'); plt.tight_layout()
plt.savefig('/tmp/nb4_rho.png', dpi=100); plt.show()
for r, it in zip(rho_values, iters_needed):
    print(f"  ρ = {r:5.1f}: {it:4d} iterations")


## 7. Key Insights for the Thesis

1. **ADMM converges to the centralized optimum** for convex portfolio coordination. The Jacobi ADMM variant enables fully parallel site updates.

2. **Privacy preservation**: sites share only net grid exchange — PV, load, battery data stay private (GDPR compliance).

3. **Dual variable $y(t)$ as internal transfer price**: represents the marginal cost of portfolio-level grid exchange at hour $t$. This is the mechanism for **benefit sharing** between BRP and prosumers.

4. **ρ tuning** significantly affects convergence speed. Adaptive ρ (Boyd et al., §3.4.1) is recommended. Typical range: $\rho \in [0.1, 10]$ depending on problem conditioning.

5. **Practical deployment**: 
   - Site updates have closed-form solutions (sub-millisecond)
   - Communication: ~1 KB per iteration (negligible)
   - Can be integrated into MPC with warm-starting for real-time operation

6. **Limitation**: the convex QP model is a simplification. Real BESS scheduling has integer variables (charge/discharge exclusion) — for MILP sub-problems, convergence is not guaranteed. Practical remedies: LP relaxation, quadratic regularization, or use centralized MILP for small portfolios and ADMM for large ones.

---

*Next: Notebook 5 implements MPC for real-time balancing.*
